# Data Microstructure

In [ ]:
# General TODOs:

# TODO: ...

# General NOTEs:

# NOTE: ...

# Setup

## Imports

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from thesis_project import (
    config,
)
from thesis_project.collectors import (
    run_collectors_double,
)
from thesis_project.utils.save import (
    save_table_image,
)

## Paths

In [ ]:
TICKER = "fbtp"
NOTEBOOK = "data-microstructure"

# Lobs directories
LOB_DIR = config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

In [ ]:
# Sample date for quick analysis
year = "2022"
month = "09"
day = "07"
date_str = f"{year}_{month}_{day}"
fut_filename = f"futures_lob_freq_1s_{date_str}.parquet"
ctd_filename = f"ctd_lob_freq_1s_{date_str}.parquet"

## Utilities

### Plotting

# Main

## Common

In [ ]:
# Common data
cf_df = pd.read_csv(
    LOB_DIR / "daily_cf.csv", parse_dates=["Date"], date_format="%Y-%m-%d", index_col="Date"
)

ctd_meta_df = pd.read_csv(
    LOB_DIR / "ctd_metadata.csv", parse_dates=["Issue Date", "Maturity Date", "First Coupon Date"]
)

fut_meta_df = pd.read_csv(
    LOB_DIR / "fut_metadata.csv", parse_dates=["Last Trading Date", "Delivery Date"]
)

ctd_switch_df = pd.read_csv(LOB_DIR / "ctd_switch.csv", parse_dates=["CTD Switch Date"])

## Gross Basis

In [ ]:
from thesis_project.collectors import compute_gross_basis

context = {
    "CF": cf_df,
}

results = run_collectors_double(
    LOB_FUT_DIR,
    LOB_CTD_DIR,
    extractors={"Basis": compute_gross_basis},
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.CTD_MRKT_OPEN,
    end_time=config.CTD_MRKT_CLOSE,
    context=context,
)

In [ ]:
gross_basis = results["Basis"]

# start_date = gross_basis.index[0].strftime("%Y-%m-%d").replace("-", "_")
# end_date = gross_basis.index[-1].strftime("%Y-%m-%d").replace("-", "_")
# filename = config.SERIES_FILENAME_TEMPLATE.format(series_name = "gross_basis", freq="1s", start_date=start_date, end_date=end_date, ext="parquet")
# gross_basis.to_parquet(OUT_INT_DIR / "basis" / filename)

## IRR

In [ ]:
from thesis_project.collectors import compute_implied_repo

context = {
    "CF": cf_df,
    "CTD Metadata": ctd_meta_df,
    "FUT Metadata": fut_meta_df,
    "Frequency": "1min",
    "Notional": 100.0,
}

results = run_collectors_double(
    LOB_FUT_DIR,
    LOB_CTD_DIR,
    extractors={"IRR": compute_implied_repo},
    start_year=config.FBTP_START_YEAR,
    end_year=config.FBTP_END_YEAR,
    start_time=config.CTD_MRKT_OPEN,
    end_time=config.CTD_MRKT_CLOSE,
    context=context,
)

In [ ]:
irr = results["IRR"]
print(
    f"Number of NaNs: {irr['ImpliedRepo'].isna().sum()} ({irr['ImpliedRepo'].isna().mean() * 100:.2f}%)"
)
display(irr)

In [ ]:
stats = {
    "mean": irr["ImpliedRepo"].mean(),
    "std": irr["ImpliedRepo"].std(),
    "skew": irr["ImpliedRepo"].skew(),
    "kurtosis": irr["ImpliedRepo"].kurtosis(),
    "min": irr["ImpliedRepo"].min(),
    "max": irr["ImpliedRepo"].max(),
    "p01": irr["ImpliedRepo"].quantile(0.01),
    "p05": irr["ImpliedRepo"].quantile(0.05),
    "median": irr["ImpliedRepo"].median(),
    "p95": irr["ImpliedRepo"].quantile(0.95),
    "p99": irr["ImpliedRepo"].quantile(0.99),
}

stats = pd.DataFrame(stats, index=["Implied Repo Rate"]).T


def make_pretty(styler):
    styler.set_caption("Implied Repo Rate Summary".upper())
    styler.format(precision=3)
    styler.set_table_styles(
        [
            {"selector": "th, td", "props": [("border", "1px solid black")]},
            {"selector": "table", "props": [("border-collapse", "collapse")]},
        ]
    )
    return styler


display(stats.style.pipe(make_pretty))
save_table_image(stats.style.pipe(make_pretty), OUT_TAB_DIR, "irr_summary")

In [ ]:
last_trading_dates = pd.to_datetime(fut_meta_df["Last Trading Date"]).dt.tz_localize(irr.index.tz)
ctd_switch_dates = pd.to_datetime(ctd_switch_df["CTD Switch Date"]).dt.tz_localize(irr.index.tz)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(irr.index, irr["ImpliedRepo"])
ax.set_title("Implied Repo Rate over Time (1min)")
ax.set_xlabel("Time")
ax.set_ylabel("IRR")
ax.grid(True, axis="y", alpha=0.3)
ax.set_xlim(irr.index[0] - pd.Timedelta("30d"), irr.index[-1] + pd.Timedelta("30d"))

for d in last_trading_dates:
    if d.date() >= irr.index[0].date() and d.date() <= irr.index[-1].date():
        ax.axvline(d, color="red", linestyle="--", alpha=0.5)

# for d in ctd_switch_dates:
#     if d.date() >= irr.index[0].date() and d.date() <= irr.index[-1].date():
#         ax.axvline(d, color="blue", linestyle="--", alpha=0.5, label="CTD Switch Date")

ax.legend(
    [
        "Implied Repo Rate",
        "Last Trading Date",
        # "CTD Switch Date"
    ]
)

fig.tight_layout()
fig.savefig(OUT_FIG_DIR / "irr.png", dpi=300, format="png")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# --- prepare dates (same as yours) ---
last_trading_dates = pd.to_datetime(fut_meta_df["Last Trading Date"]).dt.tz_localize(irr.index.tz)

ctd_switch_dates = pd.to_datetime(ctd_switch_df["CTD Switch Date"]).dt.tz_localize(irr.index.tz)

# --- group by year ---
irr["year"] = irr.index.year

for year, df in irr.groupby("year"):
    fig, ax = plt.subplots(figsize=(10, 6))

    # main series
    ax.plot(df.index, df["ImpliedRepo"], linewidth=1)

    ax.set_title(f"Implied Repo Rate (1min) - {year}")
    ax.set_xlabel("Time")
    ax.set_ylabel("IRR")

    ax.grid(True, axis="y", linestyle="--", alpha=0.4)
    ax.grid(False, axis="x")

    # x limits with padding
    ax.set_xlim(df.index[0] - pd.Timedelta("3d"), df.index[-1] + pd.Timedelta("1d"))

    # --- vertical lines: filter per year ---
    for d in last_trading_dates:
        if df.index[0] <= d <= df.index[-1]:
            ax.axvline(d, color="red", linestyle="--", alpha=0.5)

    # optional CTD switch
    for d in ctd_switch_dates:
        if df.index[0] <= d <= df.index[-1]:
            ax.axvline(d, color="blue", linestyle="--", alpha=0.5)

    # clean legend (avoid duplicates)
    ax.plot([], [], color="red", linestyle="--", alpha=0.7, label="Last Trading Date")
    ax.plot([], [], color="blue", linestyle="--", alpha=0.7, label="CTD Switch Date")
    ax.plot([], [], color="C0", label="Implied Repo Rate")

    ax.legend(frameon=False)

    # remove top/right spines
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    fig.tight_layout()

    # save per year
    fig.savefig(OUT_FIG_DIR / f"irr_{year}.png", dpi=300, format="png")
    plt.show()

    plt.close(fig)  # prevents memory issues

In [ ]:
data = irr["ImpliedRepo"].dropna()

fig, ax = plt.subplots(figsize=(10, 6))

# histogram
weights = pd.Series(1.0, index=data.index) / len(data)
ax.hist(data, bins=60, alpha=0.85, edgecolor="white", linewidth=0.7, weights=weights)

# titles and labels
ax.set_title("Implied Repo Rate Distribution (1min)")
ax.set_xlabel("Implied Repo Rate")
ax.set_ylabel("Frequency (%)")

# cleaner grid
ax.grid(True, axis="y", alpha=0.3)
ax.grid(False, axis="x")

# Ticks
ax.xaxis.set_major_locator(plt.MaxNLocator(20))
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Y axis format as percentage
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

# remove top/right spines
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout()
fig.savefig(OUT_FIG_DIR / "irr_histogram.png", dpi=300, format="png")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = irr["ImpliedRepo"].dropna()

# IQR filter
q1, q3 = np.percentile(data, [5, 95])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

filtered = data[(data >= lower) & (data <= upper)]

fig, ax = plt.subplots(figsize=(10, 6))

weights = pd.Series(1.0, index=filtered.index) / len(filtered)
ax.hist(filtered, bins=100, alpha=0.85, edgecolor="white", linewidth=0.7, weights=weights)

ax.set_title("Implied Repo Rate Distribution (1min) (5-95 Percentile)")
ax.set_xlabel("Implied Repo Rate")
ax.set_ylabel("Frequency (%)")

ax.grid(True, axis="y", alpha=0.3)
ax.grid(False, axis="x")

# Ticks
ax.xaxis.set_major_locator(plt.MaxNLocator(20))
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Y axis format as percentage
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

fig.tight_layout()
fig.savefig(OUT_FIG_DIR / "irr_histogram_cut.png", dpi=300, format="png")
plt.show()